# Project 5 — Marginal sensitivities across all 35 SB35 parameters (Tier 1)

**Goal.** For each of the 35 SB35 parameters, sweep across its full
training range with everything else held at the **corrected fiducial**
(idx-14 = 0).  Same 20 CV halos as Project 4.  This establishes the
*marginal* picture: which parameters move BIND's output, with what sign,
and which it has effectively marginalised out.

Per-halo cost: 35 × 7 = 245 generations.  Across 20 halos: 4 900
generations, ~10 min on a single A100.

**Diagnostics:**
1. **Sign matrix** (35 params × 4 observables): sign of $\Delta\mathcal{O}$
   from $\theta_{\rm min}$ to $\theta_{\rm max}$, shaded by relative
   response amplitude.
2. **Response-amplitude ranking**: sorted bar chart of total relative
   amplitude per parameter.  Identifies the parameters BIND is most/least
   sensitive to.
3. **Monotonicity score**: Spearman correlation between $\theta$ and
   $\mathcal{O}$ — flags non-monotonic responses (potential turnaround
   physics).
4. **Physics pass/fail** for the parameters with well-established sign
   expectations.

Pre-registered physics expectations:

| Param | Observable | Expected $\Delta$ |
|---|---|---|
| $A_{\rm SN1}$ ↑ | $M_\star$ | $\downarrow$ (winds suppress SF) |
| $A_{\rm AGN1}$ ↑ | $\Sigma_{\rm gas, central}$ | $\downarrow$ (AGN evacuates centre) |
| $A_{\rm SN2}$ ↑ | $M_\star$ | $\downarrow$ (faster winds, similar to A_SN1) |
| $A_{\rm AGN2}$ ↑ | $\Sigma_{\rm gas, central}$ | $\downarrow$ |
| $\sigma_8$ ↑ | $M_\star$ | $\uparrow$ (earlier collapse, more time for SF) |
| $\Omega_b$ ↑ | $f_{\rm gas}$ | $\uparrow$ (more baryons available as gas) |
| $\Omega_m$ ↑ | $M_\star$ | $\uparrow$ (more matter, more accretion) |

## 0. Setup

In [ ]:
import sys
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from scipy.stats import spearmanr

ROOT = Path('/mnt/home/mlee1/vdm_bind2')
sys.path.insert(0, str(ROOT))
from metrics import radial_profile
from data import NormStats
from train import FlowMatchingLit
from test_suite.pipeline import normalize_cutout, _denormalize_to_physical

plt.rcParams.update({
    'font.size': 10, 'font.family': 'serif', 'mathtext.fontset': 'cm',
    'figure.dpi': 110, 'savefig.bbox': 'tight',
})

SUITE_ROOT = Path('/mnt/home/mlee1/ceph/fm_testsuite')
RUN_DIR    = Path('/mnt/home/mlee1/ceph/fm_runs/fm_two_head')
CKPT       = RUN_DIR / 'checkpoints' / 'last.ckpt'
NORM_STATS = RUN_DIR / 'norm_stats.npz'
PARAM_META = '/mnt/home/mlee1/Sims/IllustrisTNG_extras/L50n512/SB35/SB35_param_minmax.csv'

SNAP, MASS_TAG, MODEL_NAME = 'snap_090', 'mass_threshold_1p000e13', 'fm_two_head'
PATCH_PIX, N_RBINS = 128, 32
N_PARAMS = 35

CACHE_DIR = ROOT / 'analysis_physics_cache'
CACHE_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / 'paper_figures'
FIG_DIR.mkdir(exist_ok=True)

PROJ5_CACHE = CACHE_DIR / f'proj5_marginals_{MODEL_NAME}.npz'
BASE_CACHE  = CACHE_DIR / f'halo_features_{MODEL_NAME}.npz'

N_GRID    = 7         # values per parameter sweep
N_HALOS   = 20
N_BINS_M  = 4
N_STEPS   = 50
BATCH_SIZE = 49       # chunk the 245-per-halo generations
USE_AMP   = True
RNG_SEED  = 17        # match project4 → same 20 halos

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '  Checkpoint:', CKPT.name)

## 1. Parameter metadata (with fiducial override)

In [ ]:
param_meta = pd.read_csv(PARAM_META)
PARAM_NAMES = {i: n for i, n in enumerate(param_meta['ParamName'])}
PARAM_LOG   = {i: bool(v) for i, v in enumerate(param_meta['LogFlag'])}
PARAM_FID   = {i: float(v) for i, v in enumerate(param_meta['FiducialVal'])}
PARAM_MIN   = {i: float(v) for i, v in enumerate(param_meta['MinVal'])}
PARAM_MAX   = {i: float(v) for i, v in enumerate(param_meta['MaxVal'])}

FIDUCIAL_OVERRIDES = {14: 0.0}    # VariableWindSpecMomentum (CV-stored=2000, CSV-fid=0)

def apply_fiducial_overrides(p: np.ndarray) -> np.ndarray:
    out = p.astype(np.float32).copy()
    for idx, val in FIDUCIAL_OVERRIDES.items():
        out[idx] = val
    return out

def param_grid(idx, n=N_GRID):
    lo, hi = PARAM_MIN[idx], PARAM_MAX[idx]
    return np.geomspace(lo, hi, n) if PARAM_LOG[idx] else np.linspace(lo, hi, n)

# Friendly LaTeX labels for the well-known parameters; rest fall back to ParamName
PRETTY = {
    0:  r'$\Omega_m$', 1:  r'$\sigma_8$',
    2:  r'$A_{\rm SN1}$', 3:  r'$A_{\rm AGN1}$',
    4:  r'$A_{\rm SN2}$', 5:  r'$A_{\rm AGN2}$',
    6:  r'$\Omega_b$',     7:  r'$h$',
    8:  r'$n_s$',
}
def label(i):
    return PRETTY.get(i, PARAM_NAMES[i])

print('Param table head:')
print(param_meta[['ParamName','LogFlag','FiducialVal','MinVal','MaxVal']].head(10))
print(f'\nApplying overrides: {FIDUCIAL_OVERRIDES}')

## 2. Halo selection — same 20 CV halos as Project 4 (RNG_SEED=17)

In [ ]:
_base = np.load(BASE_CACHE, allow_pickle=False)
_cv = _base['suite'] == 'CV'
cv_logM   = _base['logM'][_cv]
cv_simid  = _base['sim_id'][_cv]

rng = np.random.default_rng(RNG_SEED)
edges = np.quantile(cv_logM, np.linspace(0, 1, N_BINS_M + 1))
edges[0] -= 1e-6; edges[-1] += 1e-6
bin_idx = np.digitize(cv_logM, edges) - 1
per_bin = N_HALOS // N_BINS_M

selected_local = []
for b in range(N_BINS_M):
    cand = np.where(bin_idx == b)[0]
    pick = rng.choice(cand, size=min(per_bin, len(cand)), replace=False)
    selected_local.append(pick)
selected_local = np.concatenate(selected_local)

selected_meta = []
for li in selected_local:
    sid = str(cv_simid[li])
    rows_in_sim = np.where(cv_simid == sid)[0]
    pos_in_sim = int(np.where(rows_in_sim == li)[0][0])
    selected_meta.append(dict(sim_id=sid, pos_in_sim=pos_in_sim,
                              logM=float(cv_logM[li]), mass_bin=int(bin_idx[li])))
selected_df = pd.DataFrame(selected_meta).sort_values(['sim_id', 'pos_in_sim']).reset_index(drop=True)
print(f'Selected {len(selected_df)} halos across {selected_df["sim_id"].nunique()} CV sims')

## 3. Generation: 35 × 7 sweeps per halo

Per halo we build a (245, 35) param matrix where each block of 7 rows
varies one parameter across its grid with everything else at fiducial.
We then run that matrix in chunks of 49 through `fm.sample`.

Output cache: `(n_halos, 35, 7, 3, 128, 128)` ≈ 1.5 GB.

In [ ]:
def load_model():
    norm_stats = NormStats.load(NORM_STATS)
    lit = FlowMatchingLit.load_from_checkpoint(CKPT, map_location=device)
    lit.eval(); lit.to(device)
    return norm_stats, lit.fm

def _load_sim_arrays(sim_id: str):
    base = SUITE_ROOT / 'CV' / sim_id / SNAP
    cat  = np.load(base / MASS_TAG / 'halo_catalog.npz')
    cuts = np.load(base / MASS_TAG / 'halo_cutouts.npz')
    return cat, cuts

def make_marginal_param_matrix(base_params: np.ndarray) -> np.ndarray:
    """Returns (35*7, 35). Block i (rows i*7:(i+1)*7) sweeps idx i."""
    grids = [param_grid(i) for i in range(N_PARAMS)]
    M = np.tile(base_params.astype(np.float32), (N_PARAMS * N_GRID, 1))
    for i in range(N_PARAMS):
        M[i * N_GRID:(i + 1) * N_GRID, i] = grids[i]
    return M, grids

def _normalize_grid_params(M: np.ndarray, ns) -> np.ndarray:
    p = M.astype(np.float64)
    p = np.where(ns.param_log_flag == 1, np.log10(np.maximum(p, 1e-30)), p)
    rang = ns.param_max - ns.param_min
    return ((p - ns.param_min) / (rang + 1e-8)).astype(np.float32)

def generate_marginals_for_halo(fm, ns, hc, base_params, n_steps, batch_size):
    M, grids = make_marginal_param_matrix(base_params)
    cond_n, ls_n, _ = normalize_cutout(hc, ns, M[0])
    params_n = _normalize_grid_params(M, ns)
    K = len(M)
    out = []
    amp_ctx = (torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16)
               if USE_AMP and device.type == 'cuda' else nullcontext())
    with torch.no_grad():
        for s in range(0, K, batch_size):
            B = min(batch_size, K - s)
            cb = torch.from_numpy(cond_n[None].repeat(B, axis=0)).to(device)
            lb = torch.from_numpy(ls_n[None].repeat(B, axis=0)).to(device)
            pb = torch.from_numpy(params_n[s:s + B]).to(device)
            with amp_ctx:
                gen = fm.sample(cb, lb, pb, n_steps=n_steps)
            out.append(gen.float().cpu().numpy().astype(np.float32))
    gen_K = np.concatenate(out, axis=0)                # (245, C, H, W)
    physical = _denormalize_to_physical(gen_K, ns)     # (245, 3, H, W)
    return physical.reshape(N_PARAMS, N_GRID, 3, PATCH_PIX, PATCH_PIX), grids

def build_marginals_cache(force=False):
    if PROJ5_CACHE.exists() and not force:
        print('Loading', PROJ5_CACHE)
        z = np.load(PROJ5_CACHE, allow_pickle=False)
        return {k: z[k] for k in z.files}

    norm_stats, fm = load_model()
    n_h = len(selected_df)
    patches_all = np.zeros((n_h, N_PARAMS, N_GRID, 3, PATCH_PIX, PATCH_PIX), dtype=np.float32)
    base_params_per_halo = []

    last_sid = None
    grids_ref = None
    for hi, rec in tqdm(list(selected_df.iterrows()), desc='Halos'):
        sid = rec['sim_id']
        if sid != last_sid:
            cat, cuts = _load_sim_arrays(sid)
            last_sid = sid
        pos = int(rec['pos_in_sim'])
        hc = {'condition': cuts['condition'][pos], 'large_scale': cuts['large_scale'][pos]}
        base_p = apply_fiducial_overrides(np.asarray(cat['params'][pos], dtype=np.float32))
        base_params_per_halo.append(base_p.copy())
        torch.manual_seed(RNG_SEED + hi)
        patches, grids = generate_marginals_for_halo(
            fm, norm_stats, hc, base_p, n_steps=N_STEPS, batch_size=BATCH_SIZE)
        patches_all[hi] = patches
        if grids_ref is None: grids_ref = grids

    out = dict(
        patches=patches_all,
        param_grids=np.stack([np.asarray(g, dtype=np.float32) for g in grids_ref]),  # (35, 7)
        sim_id=np.asarray(selected_df['sim_id'].values, dtype='U16'),
        pos_in_sim=np.asarray(selected_df['pos_in_sim'].values, dtype=np.int32),
        logM=np.asarray(selected_df['logM'].values, dtype=np.float32),
        mass_bin=np.asarray(selected_df['mass_bin'].values, dtype=np.int32),
        base_params=np.stack(base_params_per_halo).astype(np.float32),
    )
    np.savez_compressed(PROJ5_CACHE, **out)
    print('Wrote', PROJ5_CACHE, f'  ({PROJ5_CACHE.stat().st_size/1e6:.0f} MB)')
    return out

proj5 = build_marginals_cache()
print('patches shape:', proj5['patches'].shape, '(halo, param, value, channel, H, W)')

## 4. Observables (same 4 scalars as Project 4)

In [ ]:
_yy, _xx = np.mgrid[0:PATCH_PIX, 0:PATCH_PIX] - PATCH_PIX / 2
_rr = np.sqrt(_xx ** 2 + _yy ** 2)
_bins_pix = np.linspace(0, PATCH_PIX / 2, N_RBINS + 1)
_bin_counts = np.array([((_rr >= _bins_pix[k]) & (_rr < _bins_pix[k+1])).sum()
                         for k in range(N_RBINS)], dtype=np.float64)
INNER_BINS = 8
CENTRAL_PIX = 16

def _radial(maps2d):
    out = np.empty((len(maps2d), N_RBINS))
    for i, m in enumerate(maps2d):
        _, p = radial_profile(m, n_bins=N_RBINS)
        out[i] = p
    return out

OBS_NAMES = ['M_star', 'sigma_gas_central', 'f_gas', 'compact_star']
OBS_LABELS = {
    'M_star':            r'$M_\star$',
    'sigma_gas_central': r'$\Sigma_{\rm gas, central}$',
    'f_gas':             r'$f_{\rm gas}$',
    'compact_star':      r'compact$_\star$',
}

def compute_obs(patches_NCHW: np.ndarray) -> dict:
    dm, gas, star = patches_NCHW[:, 0], patches_NCHW[:, 1], patches_NCHW[:, 2]
    M_DM, M_gas, M_star = dm.sum(axis=(1, 2)), gas.sum(axis=(1, 2)), star.sum(axis=(1, 2))
    M_tot = M_DM + M_gas + M_star + 1e-30
    half = PATCH_PIX // 2
    s, e = half - CENTRAL_PIX // 2, half + CENTRAL_PIX // 2
    sigma_gas_central = gas[:, s:e, s:e].mean(axis=(1, 2))
    p_star = _radial(star)
    enc_star = (p_star * _bin_counts).cumsum(axis=1)
    compact_star = enc_star[:, INNER_BINS - 1] / (enc_star[:, -1] + 1e-30)
    return dict(M_star=M_star, sigma_gas_central=sigma_gas_central,
                f_gas=M_gas / M_tot, compact_star=compact_star)

n_h = proj5['patches'].shape[0]
OBS = {n: np.zeros((n_h, N_PARAMS, N_GRID), dtype=np.float64) for n in OBS_NAMES}
for hi in tqdm(range(n_h), desc='Halos'):
    flat = proj5['patches'][hi].reshape(N_PARAMS * N_GRID, 3, PATCH_PIX, PATCH_PIX)
    feat = compute_obs(flat)
    for n in OBS_NAMES:
        OBS[n][hi] = feat[n].reshape(N_PARAMS, N_GRID)
print('OBS shapes:', {n: v.shape for n, v in OBS.items()}, '(halo, param, value)')

## 5. Marginal sensitivity per (param, observable)

Per parameter $i$ and observable $\mathcal{O}$, average across halos to
get $\bar{\mathcal{O}}_i(\theta_k)$ on the 7-point grid, then compute:

- **Endpoint sign**: sign of $\bar{\mathcal{O}}_i(\theta_{\max}) - \bar{\mathcal{O}}_i(\theta_{\min})$.
- **Relative amplitude**: $(\max - \min) / |\rm mean|$ along the grid.
- **Spearman ρ**: between $\theta_k$ and $\bar{\mathcal{O}}_i(\theta_k)$ — close to ±1 ⇒ monotonic, near 0 ⇒ non-monotonic.

Non-monotonic = `|ρ| < 0.7` AND amplitude > 0.05 (i.e. real response, not
monotonic).  These flag candidate turnarounds (e.g. DM back-reaction).

In [ ]:
def _summarise(arr_NV, theta):
    """arr_NV: (N_halos, V); theta: (V,). Return per-halo mean, sign, amp, spearman."""
    mean_v = arr_NV.mean(axis=0)            # mean over halos
    end_sign = float(np.sign(mean_v[-1] - mean_v[0]))
    amp = float((mean_v.max() - mean_v.min()) / (np.abs(mean_v.mean()) + 1e-30))
    rho, _ = spearmanr(theta, mean_v)
    return mean_v, end_sign, amp, float(rho if np.isfinite(rho) else 0.0)

param_grids = proj5['param_grids']           # (35, 7) raw param values
summary_rows = []
MEAN_CURVES = {n: np.zeros((N_PARAMS, N_GRID)) for n in OBS_NAMES}
SIGN_MAT    = np.zeros((N_PARAMS, len(OBS_NAMES)))
AMP_MAT     = np.zeros((N_PARAMS, len(OBS_NAMES)))
SPEAR_MAT   = np.zeros((N_PARAMS, len(OBS_NAMES)))
for j, n in enumerate(OBS_NAMES):
    for i in range(N_PARAMS):
        mean_v, sgn, amp, rho = _summarise(OBS[n][:, i, :], param_grids[i])
        MEAN_CURVES[n][i] = mean_v
        SIGN_MAT[i, j]    = sgn
        AMP_MAT[i, j]     = amp
        SPEAR_MAT[i, j]   = rho
        summary_rows.append(dict(
            idx=i, name=PARAM_NAMES[i],
            obs=n, sign=sgn, rel_amp=amp, spearman=rho,
            non_monotonic=(abs(rho) < 0.7 and amp > 0.05),
        ))
summary = pd.DataFrame(summary_rows)
print('Top 10 most-sensitive (param, obs) pairs:')
print(summary.sort_values('rel_amp', ascending=False).head(10).to_string(index=False))
print('\nNon-monotonic responses (potential turnaround physics):')
print(summary[summary['non_monotonic']].sort_values('rel_amp', ascending=False).to_string(index=False))

## 6. Sign matrix (35 × 4)

Color: red if response is negative (∂O/∂θ < 0), blue if positive.
Intensity: log-scaled relative amplitude.  White = parameter doesn't move
the observable (BIND has effectively marginalised it out).  Hatching
marks **non-monotonic** responses (`|ρ| < 0.7` with significant
amplitude).

In [ ]:
from matplotlib.colors import TwoSlopeNorm

# signed log amplitude
logA = np.log10(np.where(AMP_MAT > 1e-4, AMP_MAT, 1e-4))     # clip to avoid -inf
M_show = SIGN_MAT * (logA - np.log10(1e-4))                    # 0..max_log_amp, signed
vmax = max(np.abs(M_show).max(), 1e-30)
norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)

fig, ax = plt.subplots(figsize=(6.5, 11.0))
im = ax.imshow(M_show, aspect='auto', cmap='RdBu_r', norm=norm)
for i in range(N_PARAMS):
    for j in range(len(OBS_NAMES)):
        non_mono = abs(SPEAR_MAT[i, j]) < 0.7 and AMP_MAT[i, j] > 0.05
        if non_mono:
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                        hatch='///', edgecolor='black', lw=0.5))
        ax.text(j, i, f"{AMP_MAT[i, j]:.2f}", ha='center', va='center',
                fontsize=6, color='black' if abs(M_show[i, j]) < 0.6 * vmax else 'white')
ax.set_xticks(range(len(OBS_NAMES)))
ax.set_xticklabels([OBS_LABELS[n] for n in OBS_NAMES], rotation=20, ha='right', fontsize=9)
ax.set_yticks(range(N_PARAMS))
ax.set_yticklabels([f'{i}: {label(i)}' for i in range(N_PARAMS)], fontsize=8)
ax.set_title('Marginal sign matrix\n(red = $\\partial \\mathcal{O}/\\partial \\theta < 0$, blue = > 0;\n'
              'cell text = relative amplitude; hatched = non-monotonic)')
fig.colorbar(im, ax=ax, fraction=0.04, label='sign × log10(amplitude+1e-4)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'proj5_sign_matrix.png', dpi=150)
fig.savefig(FIG_DIR / 'proj5_sign_matrix.pdf')
plt.show()

## 7. Response-amplitude ranking

Total relative amplitude per parameter (summed across observables): which
parameters does BIND respond to at all?  Parameters at the bottom of the
ranking are *effectively marginalised out* — either physically irrelevant
at this halo-mass range, or BIND has not learned a response.

In [ ]:
tot_amp = AMP_MAT.sum(axis=1)
order = np.argsort(tot_amp)
fig, ax = plt.subplots(figsize=(7.0, 9.5))
y = np.arange(N_PARAMS)
left = np.zeros(N_PARAMS)
for j, n in enumerate(OBS_NAMES):
    ax.barh(y, AMP_MAT[order, j], left=left, label=OBS_LABELS[n])
    left += AMP_MAT[order, j]
ax.set_yticks(y)
ax.set_yticklabels([f'{i}: {label(i)}' for i in order], fontsize=8)
ax.set_xlabel('Total relative amplitude (sum of (max-min)/mean across observables)')
ax.legend(fontsize=8, loc='lower right')
ax.set_title('Per-parameter sensitivity ranking (BIND2 fm_two_head, 20 CV halos)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'proj5_amplitude_ranking.png', dpi=150)
fig.savefig(FIG_DIR / 'proj5_amplitude_ranking.pdf')
plt.show()

## 8. Physics pass/fail for the 7 well-known sensitivities

In [ ]:
PHYSICS_EXPECT = [
    # (param_idx, obs, expected_sign, description)
    (2,  'M_star',            -1, 'A_SN1 ↑ ⇒ fewer stars'),
    (3,  'sigma_gas_central', -1, 'A_AGN1 ↑ ⇒ less central gas'),
    (4,  'M_star',            -1, 'A_SN2 ↑ ⇒ fewer stars'),
    (5,  'sigma_gas_central', -1, 'A_AGN2 ↑ ⇒ less central gas'),
    (1,  'M_star',            +1, 'σ_8 ↑ ⇒ more M_★ (earlier collapse)'),
    (6,  'f_gas',             +1, 'Ω_b ↑ ⇒ more f_gas'),
    (0,  'M_star',            +1, 'Ω_m ↑ ⇒ more accretion ⇒ more M_★'),
]
rows = []
for idx, obs, expected, desc in PHYSICS_EXPECT:
    j = OBS_NAMES.index(obs)
    actual = int(SIGN_MAT[idx, j])
    rows.append(dict(
        param_idx=idx, param=label(idx), obs=obs,
        expected=expected, actual=actual,
        rel_amp=float(AMP_MAT[idx, j]),
        spearman=float(SPEAR_MAT[idx, j]),
        pass_=bool(actual == expected),
        physics=desc,
    ))
physics_df = pd.DataFrame(rows)
n_pass = physics_df['pass_'].sum()
print(f'Physics pass: {n_pass}/{len(physics_df)}')
physics_df.round(3)

## 9. Marginal response curves (top-12 most sensitive parameters)

For the 12 parameters with largest total amplitude, plot
$\bar{\mathcal{O}}(\theta)$ vs $\theta$ (one panel per parameter, 4 lines
for the 4 observables on a shared y-axis after dividing by their fiducial
value).

In [ ]:
top12 = np.argsort(tot_amp)[::-1][:12]
fig, axes = plt.subplots(3, 4, figsize=(13.0, 8.5))
for ax, idx in zip(axes.flat, top12):
    grid = param_grids[idx]
    for n in OBS_NAMES:
        v = MEAN_CURVES[n][idx]
        ax.plot(grid, v / (np.abs(v.mean()) + 1e-30), '-o', ms=3, label=OBS_LABELS[n])
    if PARAM_LOG[idx]: ax.set_xscale('log')
    ax.set_xlabel(label(idx))
    ax.set_ylabel(r'$\mathcal{O}/\langle\mathcal{O}\rangle$')
    ax.axhline(1.0, color='k', lw=0.5, ls='--')
    ax.set_title(f'idx {idx}: {PARAM_NAMES[idx][:25]}', fontsize=9)
    ax.legend(fontsize=6, loc='best')
fig.suptitle('Top-12 most sensitive marginal responses (BIND2 fm_two_head, 20 CV halos)', fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / 'proj5_top12_curves.png', dpi=150)
fig.savefig(FIG_DIR / 'proj5_top12_curves.pdf')
plt.show()

## 10. Interpretation cheatsheet

- **Cell with high amplitude + correct physical sign**: BIND is sensitive
  to this parameter and gets it right.  Tally of these is the basic
  pass-rate for the section.
- **Cell with high amplitude + wrong sign**: investigate; flag in the
  paper as a known failure or shortcut.  Possible test-leak or
  mis-normalisation.
- **Cell with low amplitude (~0)**: parameter is *effectively
  marginalised out* by BIND.  Either (a) physically irrelevant at this
  mass / observable combo, or (b) under-trained — would be evident in a
  validation-loss decomposition.
- **Hatched cell (non-monotonic)**: physically interesting candidate for
  the non-monotonic-response section of the paper.  Likeliest candidates:
  $A_{\rm SN1}$ vs $q_{\rm DM}/q_{\rm DMO}$ (back-reaction turnaround)
  and $\Omega_m$ vs $f_{\rm gas}$ (compensation between halo abundance
  and baryon retention).

Once we have both Tier 1 and Tier 2 results in:
- Sort the 35 parameters by $\sum_j$ amplitude.  Note which appear in
  Project 4's pair tests (top of ranking should overlap).
- For cells failing the Tier-1 sign check, check whether they also fail
  Tier-2 if they appear in any pair — coherent failures suggest a
  systematic bias rather than statistical noise.